In [11]:
import json
import sys
from pathlib import Path

def find_repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "lib" / "inspect").is_dir():
            return p
    raise RuntimeError("找不到仓库根目录（需包含 lib/inspect），请在 isaac-dexterous-lab 下打开工作区")

ROOT = str(find_repo_root().resolve())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import importlib
import lib.inspect.catalog as catalog
import lib.inspect.live_scene as live_scene

catalog = importlib.reload(catalog)
live_scene = importlib.reload(live_scene)
connect_isaac = live_scene.connect_isaac
fetch_live_snapshot = live_scene.fetch_live_snapshot

print("仓库根目录:", ROOT)
print("Python:", sys.executable)

仓库根目录: /home/ubuntu/isaac-dexterous-lab
Python: /usr/bin/python3


In [ ]:
# 1. 本仓库已注册的场景与机器人资产
print("可用场景:", catalog.list_scene_names())
print()
print(json.dumps(catalog.summarize_all(), indent=2, ensure_ascii=False))

In [2]:
# 3. 实时场景（统一用 lib.inspect.live_scene）
SCENE = "kuka_allegro"
spec = catalog.get_scene_spec(SCENE)
robot_paths = [r["prim"] for r in spec["robots"].values()]
object_paths = ["/World/Table", "/World/Cube"]

snapshot = fetch_live_snapshot(
    robot_paths,
    object_prim_paths=object_paths,
)
print(json.dumps(snapshot, indent=2, ensure_ascii=False))

{
  "connected": true,
  "endpoint": "127.0.0.1:8766",
  "scene": {
    "status": "success",
    "message": "pong",
    "assets_root_path": "https://omniverse-content-production.s3-us-west-2.amazonaws.com/Assets/Isaac/5.1",
    "stage_path": "",
    "prim_count": 150
  },
  "simulation": {
    "status": "success",
    "timeline_state": "playing",
    "current_time": 14.283334078267217,
    "physics_dt": 0.016666666666666666
  },
  "prims": {
    "status": "success",
    "prims": [
      {
        "path": "/World/defaultGroundPlane",
        "type": "Xform"
      },
      {
        "path": "/World/Physics_Materials",
        "type": ""
      },
      {
        "path": "/World/SceneDomeLight",
        "type": "DomeLight"
      },
      {
        "path": "/World/SceneKeyLight",
        "type": "DistantLight"
      },
      {
        "path": "/World/SceneFillLight",
        "type": "DistantLight"
      },
      {
        "path": "/World/KukaAllegro",
        "type": "Xform"
      },
      

In [3]:
# 4. 暂停 / 查询 / 单接口调用示例
with connect_isaac() as sim:
    print("场景:", sim.get_scene_info())
    print("状态:", sim.get_simulation_state())
    # sim.pause()   # 暂停 Timeline
    # sim.play()    # 恢复
    robot = robot_paths[0]
    print("关节位置:", sim.get_joint_positions(robot))

场景: {'status': 'success', 'message': 'pong', 'assets_root_path': 'https://omniverse-content-production.s3-us-west-2.amazonaws.com/Assets/Isaac/5.1', 'stage_path': '', 'prim_count': 150}
状态: {'status': 'success', 'timeline_state': 'playing', 'current_time': 18.816667648032308, 'physics_dt': 0.016666666666666666}
关节位置: {'status': 'success', 'joint_positions': [0.0006907253991812468, -0.5058321356773376, 0.7099174857139587, 1.6187671422958374, -1.517532229423523, -1.3235313892364502, 0.018803270533680916, 0.00878173764795065, 0.021438127383589745, 0.025030991062521935, 1.4579975605010986, 0.8831437230110168, 0.8764146566390991, 0.8772892951965332, 0.8861037492752075, 0.8692561984062195, 0.871893584728241, 0.8762580156326294, 0.6776787638664246, 0.8974981904029846, 0.8941134810447693, 0.8934583067893982, 0.6942617297172546]}


In [4]:
# 5. 单 Prim 查询（需要时用 get_prim_info）
with connect_isaac() as sim:
    for path in object_paths + robot_paths:
        print(path, "→", json.dumps(sim.get_prim_info(path), ensure_ascii=False))

/World/Table → {"status": "success", "path": "/World/Table", "type": "Cube", "transform": {"position": [0.550000011920929, 0.0, 0.7300000190734863]}, "children": [], "actual_size": [1.0, 0.699999988079071, 0.05999999865889549]}
/World/Cube → {"status": "success", "path": "/World/Cube", "type": "Cube", "transform": {"position": [0.5500006079673767, 3.9687586195213953e-07, 0.7799999117851257]}, "children": [], "actual_size": [0.03999999910593033, 0.03999999910593033, 0.03999999910593033]}
/World/KukaAllegro → {"status": "success", "path": "/World/KukaAllegro", "type": "Xform", "transform": {"position": [0.0, 0.0, 0.0]}, "children": ["/World/KukaAllegro/ee_link", "/World/KukaAllegro/iiwa7_link_7", "/World/KukaAllegro/iiwa7_link_0", "/World/KukaAllegro/root_joint", "/World/KukaAllegro/iiwa7_link_1", "/World/KukaAllegro/iiwa7_link_2", "/World/KukaAllegro/iiwa7_link_3", "/World/KukaAllegro/iiwa7_link_4", "/World/KukaAllegro/iiwa7_link_5", "/World/KukaAllegro/iiwa7_link_6"]}


In [7]:
with connect_isaac() as sim:
    print(sim.get_scene_info())

{'status': 'success', 'message': 'pong', 'assets_root_path': 'https://omniverse-content-production.s3-us-west-2.amazonaws.com/Assets/Isaac/5.1', 'stage_path': '', 'prim_count': 150}


In [10]:
with connect_isaac() as sim:
    result = sim.list_prims("/World")
    prims = result["prims"]
    print("/World 下有", len(prims), "个 Prim:")
    for p in prims:
        print(f"{p['path']} ({p['type']})\n")


/World 下有 9 个 Prim:
/World/defaultGroundPlane (Xform)

/World/Physics_Materials ()

/World/SceneDomeLight (DomeLight)

/World/SceneKeyLight (DistantLight)

/World/SceneFillLight (DistantLight)

/World/KukaAllegro (Xform)

/World/Table (Cube)

/World/Looks ()

/World/Cube (Cube)



In [11]:
with connect_isaac() as sim:
  robots = sim.list_robots()           # 资产库里的机器人列表
  print("资产库机器人数:", robots.get("robot_count"))

  # 当前场景里已加载的机器人（看 /World 下的 articulation）
  world = sim.list_prims("/World")["prims"]
  robot_like = [p for p in world if "Kuka" in p["path"] or "Franka" in p["path"]]
  print("场景里机器人 Prim:", robot_like)

资产库机器人数: 107
场景里机器人 Prim: [{'path': '/World/KukaAllegro', 'type': 'Xform'}]


In [12]:
# 按关节名打印 KukaAllegro 信息
ROBOT = "/World/KukaAllegro"

with connect_isaac() as sim:
    joints = sim.robot_joints_by_name(ROBOT)
    print(f"DOF 数: {len(joints)}\n")
    for name, data in joints.items():
        pos = data.get("position_rad")
        pos_str = f"{pos:+.4f} rad" if pos is not None else "N/A"
        lower = data.get("lower")
        upper = data.get("upper")
        limit_str = f"[{lower}, {upper}] {data.get('units', '')}" if lower is not None else ""
        print(f"{name}")
        print(f"  index={data.get('index')}  type={data.get('type')}  position={pos_str}")
        if limit_str:
            print(f"  limits={limit_str}")
        if "stiffness" in data or "damping" in data:
            print(f"  stiffness={data.get('stiffness')}  damping={data.get('damping')}")
        if "target_position" in data:
            print(f"  target={data.get('target_position')}  actual={data.get('actual_position')}")
        print()


DOF 数: 23

iiwa7_joint_1
  index=0  type=revolute  position=-0.0002 rad
  limits=[-169.99998474121094, 169.99998474121094] degrees
  stiffness=10000000.0  damping=0.0
  target=0.0  actual=-0.00022908022219780833

iiwa7_joint_2
  index=1  type=revolute  position=-0.4461 rad
  limits=[-120.0, 120.0] degrees
  stiffness=10000000.0  damping=0.0
  target=0.0  actual=-0.44613730907440186

iiwa7_joint_3
  index=2  type=revolute  position=+0.7257 rad
  limits=[-169.99998474121094, 169.99998474121094] degrees
  stiffness=10000000.0  damping=0.5
  target=0.7853999733924866  actual=0.7256477475166321

iiwa7_joint_4
  index=3  type=revolute  position=+1.8079 rad
  limits=[-120.0, 120.0] degrees
  stiffness=10000000.0  damping=0.5
  target=1.5707999467849731  actual=1.8079314231872559

iiwa7_joint_5
  index=4  type=revolute  position=-1.5152 rad
  limits=[-169.99998474121094, 169.99998474121094] degrees
  stiffness=10000000.0  damping=0.5
  target=-1.5707999467849731  actual=-1.5152482986450195

ii